In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import numpy as np
import MDAnalysis as mda
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from code_libraries.counts_rdfs_3 import *
from code_libraries.predict_acyl_chains_NN_by_chain import *
from scipy import integrate

In [ ]:
def convolve_positions(pos_array, n_point):

    n_frames, n_lipids, n_coords = pos_array.shape
    n_reduced = n_frames - n_point + 1
    kernel = np.ones(n_point) / n_point

    convolved = np.empty([n_reduced, n_lipids, n_coords])

    for j in range(n_lipids):
        for c in range(n_coords):
            convolved[:, j, c] = np.convolve(pos_array[:, j, c], kernel, mode="valid")

    return convolved

In [ ]:
start_frame , stop_frame = 0 , 4000
sys_type = "sym"
sys_name_75 = "dopc_75"

In [ ]:
data_array_dir = "../heavy_data_NO_GIT/data_arrays_by_chain_xgb/"

## 17 pt

In [ ]:
dopc_75_box_dim = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_17pt_box_dim.npy"))
dopc_75_17pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_17pt_prediction_labels.npy") , allow_pickle=True)
dopc_75_17pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn2_ft_last_17pt_prediction_labels.npy") , allow_pickle=True)
dopc_75_box_dim = dopc_75_box_dim[: , :2]

sys_75_box_dim = dopc_75_box_dim
sys_75_box_dim = sys_75_box_dim[: , :2]
sys_75_box_dim = np.mean(sys_75_box_dim , axis = 0)

In [ ]:
dopc_75_17pt_sn1_pred_labels_lower  , dopc_75_17pt_sn1_pred_labels_upper = dopc_75_17pt_sn1_prediction_labels
dopc_75_17pt_sn2_pred_labels_lower  , dopc_75_17pt_sn2_pred_labels_upper = dopc_75_17pt_sn2_prediction_labels

In [ ]:
dopc_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "xgb" , "sn1" , 3 , start_frame , stop_frame).bilayer_all_dopc_data()
chol_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "xgb" , "sn1", 3 , start_frame , stop_frame).bilayer_all_chol_data()

In [ ]:
dopc_75_pos_lower , dopc_75_pos_upper = dopc_75_pos_raw[0] , dopc_75_pos_raw[1]
chol_75_pos_lower , chol_75_pos_upper = chol_75_pos_raw[0] , chol_75_pos_raw[1]

dopc_75_phos_lower = dopc_75_pos_lower[: , : , 19 , :]
dopc_75_phos_upper = dopc_75_pos_upper[: , : , 19 , :]
dopc_75_nitro_lower = dopc_75_pos_lower[: , : , 0 , :]
dopc_75_nitro_upper = dopc_75_pos_upper[: , : , 0 , :]
chol_75_oxy_lower = chol_75_pos_lower[: , : , 2 , :]
chol_75_oxy_upper = chol_75_pos_upper[: , : , 2 , :]

dopc_75_phos_lower_17pt = convolve_positions(dopc_75_phos_lower , 17)
dopc_75_phos_upper_17pt = convolve_positions(dopc_75_phos_upper , 17)
dopc_75_nitro_lower_17pt = convolve_positions(dopc_75_nitro_lower , 17)
dopc_75_nitro_upper_17pt = convolve_positions(dopc_75_nitro_upper , 17)
chol_75_oxy_lower_17pt = convolve_positions(chol_75_oxy_lower , 17)
chol_75_oxy_upper_17pt = convolve_positions(chol_75_oxy_upper , 17)

In [ ]:
pbc_distances_75_phos_chol_lower = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_phos_lower_17pt , chol_75_oxy_lower_17pt , sys_75_box_dim)
pbc_distances_75_phos_chol_upper = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_phos_upper_17pt , chol_75_oxy_upper_17pt , sys_75_box_dim)
pbc_distances_75_nitro_chol_lower = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_nitro_lower_17pt , chol_75_oxy_lower_17pt , sys_75_box_dim)
pbc_distances_75_nitro_chol_upper = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_nitro_upper_17pt , chol_75_oxy_upper_17pt , sys_75_box_dim)

In [ ]:
indices_75_17pt_both_lo_lower = dopc_75_17pt_sn1_pred_labels_lower & dopc_75_17pt_sn2_pred_labels_lower
indices_75_17pt_both_lo_upper = dopc_75_17pt_sn1_pred_labels_upper & dopc_75_17pt_sn2_pred_labels_upper

indices_75_17pt_both_ld_lower = ~dopc_75_17pt_sn1_pred_labels_lower & ~dopc_75_17pt_sn2_pred_labels_lower
indices_75_17pt_both_ld_upper = ~dopc_75_17pt_sn1_pred_labels_upper & ~dopc_75_17pt_sn2_pred_labels_upper

In [ ]:
both_lo_pbc_distances_phos_chol_75_17pt_lower = predictions_cutoff_pbc(pbc_distances_75_phos_chol_lower , indices_75_17pt_both_lo_lower)[0]
both_lo_pbc_distances_phos_chol_75_17pt_upper = predictions_cutoff_pbc(pbc_distances_75_phos_chol_upper , indices_75_17pt_both_lo_upper)[0]

both_ld_pbc_distances_phos_chol_75_17pt_lower = predictions_cutoff_pbc(pbc_distances_75_phos_chol_lower , indices_75_17pt_both_ld_lower)[0]
both_ld_pbc_distances_phos_chol_75_17pt_upper = predictions_cutoff_pbc(pbc_distances_75_phos_chol_upper , indices_75_17pt_both_ld_upper)[0]


both_lo_pbc_distances_nitro_chol_75_17pt_lower = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_lower , indices_75_17pt_both_lo_lower)[0]
both_lo_pbc_distances_nitro_chol_75_17pt_upper = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_upper , indices_75_17pt_both_lo_upper)[0]

both_ld_pbc_distances_nitro_chol_75_17pt_lower = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_lower , indices_75_17pt_both_ld_lower)[0]
both_ld_pbc_distances_nitro_chol_75_17pt_upper = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_upper , indices_75_17pt_both_ld_upper)[0]

In [ ]:
dr = 0.05
radius = 2

radial_distances_75 = np.arange(0 , radius , dr)

both_lo_phos_chol_rdf_75_17pt_lower = RDF_over_all_frames_diff_species(both_lo_pbc_distances_phos_chol_75_17pt_lower , dr , radius ,sys_75_box_dim )
both_lo_phos_chol_rdf_75_17pt_upper = RDF_over_all_frames_diff_species(both_lo_pbc_distances_phos_chol_75_17pt_upper , dr , radius ,sys_75_box_dim )

both_ld_phos_chol_rdf_75_17pt_lower = RDF_over_all_frames_diff_species(both_ld_pbc_distances_phos_chol_75_17pt_lower , dr , radius ,sys_75_box_dim )
both_ld_phos_chol_rdf_75_17pt_upper = RDF_over_all_frames_diff_species(both_ld_pbc_distances_phos_chol_75_17pt_upper , dr , radius ,sys_75_box_dim )


both_lo_nitro_chol_rdf_75_17pt_lower = RDF_over_all_frames_diff_species(both_lo_pbc_distances_nitro_chol_75_17pt_lower , dr , radius ,sys_75_box_dim )
both_lo_nitro_chol_rdf_75_17pt_upper = RDF_over_all_frames_diff_species(both_lo_pbc_distances_nitro_chol_75_17pt_upper , dr , radius ,sys_75_box_dim )

both_ld_nitro_chol_rdf_75_17pt_lower = RDF_over_all_frames_diff_species(both_ld_pbc_distances_nitro_chol_75_17pt_lower , dr , radius ,sys_75_box_dim )
both_ld_nitro_chol_rdf_75_17pt_upper = RDF_over_all_frames_diff_species(both_ld_pbc_distances_nitro_chol_75_17pt_upper , dr , radius ,sys_75_box_dim )


av_75_17pt_phos_chol_both_lo_rdf = 0.5*(np.array(both_lo_phos_chol_rdf_75_17pt_lower) + np.array(both_lo_phos_chol_rdf_75_17pt_upper))
av_75_17pt_phos_chol_both_ld_rdf = 0.5*(np.array(both_ld_phos_chol_rdf_75_17pt_lower) + np.array(both_ld_phos_chol_rdf_75_17pt_upper))

av_75_17pt_nitro_chol_both_lo_rdf = 0.5*(np.array(both_lo_nitro_chol_rdf_75_17pt_lower) + np.array(both_lo_nitro_chol_rdf_75_17pt_upper))
av_75_17pt_nitro_chol_both_ld_rdf = 0.5*(np.array(both_ld_nitro_chol_rdf_75_17pt_lower) + np.array(both_ld_nitro_chol_rdf_75_17pt_upper))

In [ ]:
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_title(r'Headgroup-CHOL RDF, $\mathbf{\chi_C}$ = 0.25 -- XGB 17pt', fontsize=30, fontweight='bold')
ax.plot(radial_distances_75, av_75_17pt_phos_chol_both_lo_rdf, color="green", linewidth=3, label="Lo CHOL(O)-DOPC(P)")
ax.plot(radial_distances_75, av_75_17pt_phos_chol_both_ld_rdf, color="blue", linewidth=3, label="Ld CHOL(O)-DOPC(P)")
ax.plot(radial_distances_75, av_75_17pt_nitro_chol_both_lo_rdf, color="red", linewidth=3, label="Lo CHOL(O)-DOPC(N)")
ax.plot(radial_distances_75, av_75_17pt_nitro_chol_both_ld_rdf, color="black", linewidth=3, label="Ld CHOL(O)-DOPC(N)")

ax.axhline(y=1, color='gray', linestyle=':', linewidth=2)
ax.set_xlabel(r'$\mathbf{r \, (nm)}$', fontsize=32, fontweight='bold')
ax.set_ylabel(r'$\mathbf{g(r)}$', fontsize=32, fontweight='bold')
ax.set_xlim(0, 2.0)
ax.set_ylim(0, 1.5)
ax.set_xticks([0.5, 1.0, 1.5, 2.0])
ax.set_yticks([0.0, 0.3, 0.6, 0.9, 1.2, 1.5])
ax.tick_params(axis='both', which='major', labelsize=32, width=2.5, length=8, direction="inout")
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')
legend = ax.legend(fontsize=20, prop={'weight':'bold', 'size': 30}, frameon=True, fancybox=True,
                  loc = "lower right")
for line in legend.get_lines():
    line.set_linewidth(3.0)
plt.tight_layout()
plt.show()

### Umbrella index: area between the N-CHOL and P-CHOL g(r)

In [ ]:
integrate.trapezoid(av_75_17pt_nitro_chol_both_lo_rdf ,radial_distances_75 ) - integrate.trapezoid(av_75_17pt_phos_chol_both_lo_rdf ,radial_distances_75 )

In [ ]:
integrate.trapezoid(av_75_17pt_nitro_chol_both_ld_rdf ,radial_distances_75 ) - integrate.trapezoid(av_75_17pt_phos_chol_both_ld_rdf ,radial_distances_75 )

In [ ]:
abs_diff_75_17pt_lo_umb_index = np.abs(av_75_17pt_nitro_chol_both_lo_rdf - av_75_17pt_phos_chol_both_lo_rdf)
abs_diff_75_17pt_ld_umb_index = np.abs(av_75_17pt_nitro_chol_both_ld_rdf - av_75_17pt_phos_chol_both_ld_rdf)

In [ ]:
end_index = 40

integrate.trapezoid(abs_diff_75_17pt_lo_umb_index[:end_index] ,radial_distances_75[:end_index]) , integrate.trapezoid(abs_diff_75_17pt_ld_umb_index[:end_index] , radial_distances_75[:end_index])